# Theme 2: RAG Application, Starter Kit

This starter kit contains a complete, working **Retrieval-Augmented Generation (RAG)** application using **Gemma 4**, Google's open-weight model. Your team's task is to adapt and extend it for the problem statement you selected (2A or 2B).

**How RAG works:**

```
 Document ──► 1. split into chunks ──► 2. convert each chunk into an embedding (a list of numbers)
                                                           │
 Question ──► 3. retrieve the chunks whose embeddings are closest to the question's embedding
                                                           │
              4. give ONLY those chunks to Gemma ──► answer with page citations
```

**Models used (both open-weight):**
- `all-MiniLM-L6-v2` (sentence-transformers) creates the embeddings. It runs inside this notebook.
- `gemma-4-26b-a4b-it` (Google) writes the answers. It is accessed through the free Google AI Studio API.

**No GPU is required.** Use the default Colab runtime (CPU).

### Obtaining a free Google AI Studio API key (one per team)
1. Go to **https://aistudio.google.com/apikey** and sign in with a Google account.
2. Select **Create API key** and copy the key.
3. In Colab, select the **key icon** in the left sidebar (**Secrets**).
4. Select **Add new secret**. Set **Name** to `GOOGLE_API_KEY` and **Value** to your key. Enable **Notebook access**.

> **Important:** do not paste the key directly into a code cell. Anyone the notebook is shared with would be able to see it.
>
> **No key yet:** the notebook still runs. The retrieval stages work normally, and the application shows the passages it found in place of a written answer.

### How to use this notebook
- Run the cells **in order, from top to bottom**, the first time (select a cell and press **Shift + Enter**).
- Comments marked **STUDENT TODO** show the intended places to make changes.
- After changing a cell, re-run that cell and any cells below it that depend on it.
- If the notebook stops working as expected, select **Runtime → Restart session** and run all cells again from the top.

## Step 0: Installation (run once)

In [28]:
# Installs the required libraries. This takes about one minute.
!pip install -q pypdf sentence-transformers "gradio>=6,<7" google-genai

## Step 1: Settings

These values control how documents are divided, searched and answered. They are the main parameters to experiment with.

In [29]:
# STUDENT TODO: the Gemma 4 model that writes the answers.
#   "gemma-4-26b-a4b-it"  Mixture of Experts, 26B total / about 4B active parameters. Fast. (default)
#   "gemma-4-31b-it"      Dense, 31B parameters. More capable, slower.
GEMMA_MODEL = "gemma-4-26b-a4b-it"

# STUDENT TODO: the number of words in each chunk.
#   Small chunks (50)   more precise matches, but may lose context
#   Large chunks (300)  more context, but less precise and longer prompts
CHUNK_SIZE = 120

# STUDENT TODO: the number of words shared by neighbouring chunks, so sentences are not split.
CHUNK_OVERLAP = 30

# STUDENT TODO: the number of chunks given to Gemma for each question.
TOP_K = 3

# STUDENT TODO: the rules Gemma follows when answering. The rule to use only the context prevents invented answers.
ANSWER_RULES = """You answer questions about a document using ONLY the context passages provided.
- Each passage begins with its page number, for example [Page 2].
- After each fact you use, cite the page in brackets, for example (p. 2).
- If the answer is not in the context, reply exactly: "The answer could not be found in the document."
- Keep answers concise: 2 to 4 sentences."""

TEMPERATURE = 0.2       # a low value keeps answers consistent and close to the document

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # STUDENT TODO (optional): try "BAAI/bge-small-en-v1.5"
print("Settings configured.")

Settings configured.


## Step 2: Connect to Gemma and load the embedding model

In [30]:
import os
from google import genai
from google.genai import types, errors


def _get_secret(name):
    """Reads a secret from the Colab Secrets panel (or from an environment variable)."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)


GOOGLE_API_KEY = _get_secret("GOOGLE_API_KEY")
DEMO_MODE = not GOOGLE_API_KEY
client = None if DEMO_MODE else genai.Client(api_key=GOOGLE_API_KEY)


def _friendly_error(e):
    code = getattr(e, "code", None)
    if code in (400, 401, 403) and "key" in str(e).lower():
        return "Error: the GOOGLE_API_KEY was rejected. Check that the complete key is saved in Colab Secrets, then re-run this cell."
    if code == 429:
        return "Error: the free-tier usage limit has been reached. Wait about one minute and try again."
    if code == 404:
        return f"Error: the model '{GEMMA_MODEL}' was not found. Check the GEMMA_MODEL setting."
    return f"Error: the request to Gemma failed ({e})."


def generate(user_text, system_text=None, temperature=TEMPERATURE, demo_reply="(Demonstration mode response)"):
    """Sends one message to Gemma and returns its reply as text.
    In demonstration mode, returns `demo_reply`. If the request fails, returns a message beginning with "Error:"."""
    if DEMO_MODE:
        return demo_reply
    try:
        response = client.models.generate_content(
            model = GEMMA_MODEL,
            contents = user_text,
            config = types.GenerateContentConfig(
                system_instruction = system_text,
                temperature = temperature,
                thinking_config = types.ThinkingConfig(thinking_level="minimal"),  # faster replies
            ),
        )
        return (response.text or "").strip()
    except errors.APIError as e:
        return _friendly_error(e)
    except Exception as e:
        return f"Error: {e}"


if DEMO_MODE:
    print("Demonstration mode: no GOOGLE_API_KEY was found. Retrieval will work; answers will show the passages found.")
else:
    print(f"Connected to Google AI Studio. Model: {GEMMA_MODEL}")
    print(generate("In one sentence, what is an open-weight AI model?"))

from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer(EMBED_MODEL_NAME)   # runs on the CPU
print(f"Loaded embedding model: {EMBED_MODEL_NAME}")

Connected to Google AI Studio. Model: gemma-4-26b-a4b-it
An open-weight AI model is a machine learning model where the trained parameters (weights) are made publicly available, allowing developers to download, run, and fine-tune the model on their own hardware.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded embedding model: sentence-transformers/all-MiniLM-L6-v2


## Step 3: Sample document
A short, fictional course syllabus is provided so the application works immediately. Each item in the list is one page. Your own PDF can be uploaded in the application in Step 5.

In [31]:
# STUDENT TODO: replace with text relevant to your problem statement, or upload a PDF in the application instead.
SAMPLE_PAGES = [
    """CS 101: Introduction to Programming (Sample Syllabus). Instructor: Dr. A. Rao. Email: arao@example.edu.
    Lectures are held Monday and Wednesday, 10:00 to 11:30 AM, in Room B-204. Labs are on Friday from 2:00 to 4:00 PM
    in the Computer Lab (Room C-110). Office hours: Tuesday 3:00 to 5:00 PM, or by appointment. This course introduces
    programming using Python. No prior coding experience is required.""",

    """Grading. Weekly lab exercises: 20%. Two mini-projects: 30% (15% each). Midterm exam: 20%. Final exam: 25%.
    Class participation: 5%. A score of 90% or above earns an A, 80 to 89% a B, 70 to 79% a C, and 60 to 69% a D.
    Late submissions lose 10% per day, up to a maximum of 3 days, after which they receive zero. Each student has
    two grace days per semester that may be used to submit any lab late without penalty.""",

    """Schedule. Weeks 1-2: variables, data types, and input/output. Weeks 3-4: conditions and loops. Week 5: functions.
    Week 6: lists and dictionaries. Week 7: Midterm exam (in class, closed book). Weeks 8-9: files and error handling.
    Weeks 10-11: an introduction to APIs and working with data. Weeks 12-13: mini-project 2, a team project of up to
    3 students. Week 14: review. The final exam is a 2-hour written exam during the examination period.""",

    """Policies. Attendance: students must attend at least 75% of lectures to be eligible for the final exam.
    Academic integrity: students may discuss ideas with classmates, but all submitted code must be their own. AI
    tools may be used for learning and debugging, but their use must be stated in a comment at the top of the file.
    Laptops: bring a laptop to every lab. Students without a laptop may reserve a machine in the Computer Lab.
    Textbook: "Automate the Boring Stuff with Python" (available free online).""",
]
print(f"Sample document contains {len(SAMPLE_PAGES)} pages.")

Sample document contains 4 pages.


## Step 4: The RAG pipeline
Each function is one stage of the pipeline. This is the core of the project and the main code to extend.

In [32]:
import numpy as np
from pypdf import PdfReader

knowledge_base = {"name": None, "chunks": [], "vectors": None}  # populated by build_knowledge_base()


def read_pdf(path):
    """Returns a list of page texts from a PDF file."""
    return [page.extract_text() or "" for page in PdfReader(path).pages]


def split_into_chunks(pages):
    """STAGE 1: split each page into overlapping chunks of about CHUNK_SIZE words."""
    chunks = []
    step = max(1, CHUNK_SIZE - CHUNK_OVERLAP)
    for page_number, text in enumerate(pages, start=1):
        words = text.split()
        for start in range(0, len(words), step):
            chunk_words = words[start:start + CHUNK_SIZE]
            if len(chunk_words) > 5:  # skip very short fragments
                chunks.append({"page": page_number, "text": " ".join(chunk_words)})
            if start + CHUNK_SIZE >= len(words):
                break
    return chunks


def build_knowledge_base(pages, name):
    """STAGE 2: convert each chunk into an embedding."""
    chunks = split_into_chunks(pages)
    if not chunks:
        raise ValueError("No text was found. The PDF may contain scanned images only.")
    vectors = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True)
    knowledge_base.update(name=name, chunks=chunks, vectors=vectors)
    return f"Loaded **{name}**: {len(pages)} pages, {len(chunks)} chunks."


def retrieve(question, top_k=TOP_K):
    """STAGE 3: find the chunks whose meaning is closest to the question."""
    question_vector = embedder.encode([question], normalize_embeddings=True)[0]
    similarities = knowledge_base["vectors"] @ question_vector   # 1.0 = identical meaning
    best = np.argsort(similarities)[::-1][:top_k]
    return [{**knowledge_base["chunks"][i], "score": float(similarities[i])} for i in best]


def answer(question):
    """STAGE 4: give the retrieved chunks to Gemma and ask it to answer using only them."""
    found = retrieve(question)
    context = "\n\n".join(f"[Page {c['page']}] {c['text']}" for c in found)
    prompt = f"Context:\n{context}\n\nQuestion: {question}"
    demo = (f"**Demonstration mode:** no API key is set, so Gemma did not write an answer. Retrieval completed; "
            f"the most relevant passage is on **page {found[0]['page']}**:\n\n> {found[0]['text'][:400]}...")
    reply = generate(prompt, system_text=ANSWER_RULES, demo_reply=demo)
    return reply, found


# Quick test with the sample document.
print(build_knowledge_base(SAMPLE_PAGES, "Sample CS 101 syllabus"))
test_question = "What happens if I submit a lab late?"
reply, found = answer(test_question)
print(f"\nQuestion: {test_question}\nAnswer: {reply}\n\nRetrieved passages:")
for c in found:
    print(f"   [page {c['page']}, similarity {c['score']:.2f}] {c['text'][:90]}...")

Loaded **Sample CS 101 syllabus**: 4 pages, 4 chunks.

Question: What happens if I submit a lab late?
Answer: Late submissions lose 10% per day for up to a maximum of 3 days, after which they receive a score of zero (p. 2). However, each student is provided two grace days per semester that can be used to submit any lab late without penalty (p. 2).

Retrieved passages:
   [page 2, similarity 0.53] Grading. Weekly lab exercises: 20%. Two mini-projects: 30% (15% each). Midterm exam: 20%. ...
   [page 4, similarity 0.33] Policies. Attendance: students must attend at least 75% of lectures to be eligible for the...
   [page 3, similarity 0.28] Schedule. Weeks 1-2: variables, data types, and input/output. Weeks 3-4: conditions and lo...


## Step 5: Launch the application
Run this cell to open the application below the cell and create a **public link** (`https://....gradio.live`). Re-run it after changing any cell above.



# ============================================================
# REPLACEMENT FOR STEP 5 (paste into a NEW cell at the bottom,
# or replace the old Step 5 cell). Run Steps 0-4 first.
# Campus Regulations Assistant: multiple documents, citations
# with document names, relevance threshold, cleaner interface.
# ============================================================
import os
import numpy as np
import gradio as gr

# ---------- STUDENT TODO: your settings ----------
COLLEGE_NAME = "Your College Name"          # shown in the title
CONTACT_HINT = "the Examination Office or your department office"   # who to contact when an answer is not found
MIN_SCORE = 0.20    # below this similarity, decline WITHOUT calling Gemma. Tune it using your 15-question test.
EXAMPLE_QUESTIONS = [
    "What is the minimum attendance required to sit the end-semester examination?",
    "Can I apply for re-evaluation, and what is the deadline?",
    "What are the hostel visiting hours?",
    "Who won the 2022 World Cup?",           # keep one question the documents cannot answer
]

RULES_V2 = """You answer students' questions using ONLY the context passages provided.
- Each passage begins with its source, for example [Exam Regulations.pdf, p. 4].
- After each fact you use, cite the source in brackets, for example (Exam Regulations.pdf, p. 4).
- If the answer is not in the context, reply exactly: "The answer could not be found in the documents."
- If two passages disagree, mention both and cite both.
- Keep answers concise: 2 to 4 sentences."""


# ---------- Reading files (PDF, Markdown, text) ----------
def read_file(path):
    """Returns a list of page texts. Markdown and text files count as one page."""
    if path.lower().endswith(".pdf"):
        return read_pdf(path)                      # defined in Step 4
    return [open(path, encoding="utf-8", errors="ignore").read()]


def add_documents(files):
    """Reads every uploaded file, reports a result for EACH file, and rebuilds the knowledge base."""
    if not files:
        return "No files uploaded yet."
    all_chunks, report = [], []
    for path in files:
        name = os.path.basename(path)
        try:
            pages = read_file(path)
            chunks = split_into_chunks(pages)      # defined in Step 4
            if not chunks:
                raise ValueError("no readable text (scanned image?)")
            for c in chunks:
                c["doc"] = name
            all_chunks += chunks
            report.append(f"✅ **{name}**: {len(pages)} pages, {len(chunks)} chunks")
        except Exception as e:
            report.append(f"❌ **{name}**: {type(e).__name__}: {e}")   # the real error stays visible here
    if all_chunks:
        vectors = embedder.encode([c["text"] for c in all_chunks], normalize_embeddings=True)
        knowledge_base.update(name=f"{len(files)} documents", chunks=all_chunks, vectors=vectors)
    return "\n\n".join(report)


# ---------- The pipeline, with document names and a relevance threshold ----------
def answer_v2(question):
    if not knowledge_base["chunks"]:
        return "Upload at least one document first.", []
    found = retrieve(question)                     # defined in Step 4
    if found[0]["score"] < MIN_SCORE:              # nothing relevant: do not call Gemma at all
        return f"The answer could not be found in the documents. Please contact {CONTACT_HINT}.", found
    context = "\n\n".join(f"[{c['doc']}, p. {c['page']}] {c['text']}" for c in found)
    reply = generate(f"Context:\n{context}\n\nQuestion: {question}", system_text=RULES_V2,
                     demo_reply=f"**Demonstration mode:** no API key. Best match: {found[0]['doc']}, "
                                f"page {found[0]['page']}:\n\n> {found[0]['text'][:400]}...")
    if reply.startswith("The answer could not be found"):
        reply += f" Please contact {CONTACT_HINT}."
    return reply, found


# ---------- Interface ----------
def on_ask(question):
    if not question or not question.strip():
        raise gr.Error("Type a question first.")
    reply, found = answer_v2(question)
    chips = ""
    if found and found[0]["score"] >= MIN_SCORE:
        seen = []
        for c in found:
            label = f"{c['doc']} · p. {c['page']}"
            if label not in seen:
                seen.append(label)
        chips = "<div class='chips'>" + "".join(f"<span class='chip'>{s}</span>" for s in seen) + "</div>"
    sources = "\n\n".join(
        f"**{c['doc']}, page {c['page']}** (similarity {c['score']:.2f})\n> {c['text']}" for c in found
    ) or "No passages retrieved."
    return reply, chips, sources


CSS = """
.gradio-container {max-width: 880px !important; margin: auto;}
.chips {display: flex; flex-wrap: wrap; gap: 8px; margin: 4px 0 12px;}
.chip {background: #E6F2F0; color: #0B5D55; border: 1px solid #B7D9D4; border-radius: 6px;
       padding: 3px 10px; font-size: 0.85rem;}
#ask-btn {font-size: 1.05rem;}
"""
THEME = gr.themes.Soft(primary_hue="teal", neutral_hue="slate",
                       font=[gr.themes.GoogleFont("IBM Plex Sans"), "system-ui", "sans-serif"])

gr.close_all()
with gr.Blocks(title=f"{COLLEGE_NAME} Regulations Assistant") as demo:
    gr.Markdown(f"# {COLLEGE_NAME} Regulations Assistant\n"
                "Ask about attendance, exams, hostel and library rules. "
                "Every answer cites the document and page it came from.")
    with gr.Accordion("1. Upload the official documents", open=True):
        files_in = gr.File(label="PDF, .md or .txt files", file_count="multiple",
                           file_types=[".pdf", ".md", ".txt"], type="filepath")
        status_md = gr.Markdown("No documents loaded yet.")
    question_box = gr.Textbox(label="2. Your question", lines=2,
                              placeholder="For example: What is the minimum attendance for the end-semester exam?")
    ask_btn = gr.Button("Get answer", variant="primary", elem_id="ask-btn")
    gr.Examples(EXAMPLE_QUESTIONS, inputs=question_box, label="Try an example")
    answer_md = gr.Markdown()
    chips_html = gr.HTML()
    with gr.Accordion("Passages used for this answer", open=False):
        sources_md = gr.Markdown()

    files_in.change(add_documents, inputs=files_in, outputs=status_md)
    ask_btn.click(on_ask, inputs=question_box, outputs=[answer_md, chips_html, sources_md])
    question_box.submit(on_ask, inputs=question_box, outputs=[answer_md, chips_html, sources_md])

demo.launch(share=True, theme=THEME, css=CSS)

In [34]:

import os
import numpy as np
import gradio as gr

# ---------- STUDENT TODO: your settings ----------
COLLEGE_NAME = "upGrad School of Technology"          # shown in the title
CONTACT_HINT = "the Academic Office"   # EDIT THIS: a real office or person students can contact
MIN_SCORE = 0.20    # below this similarity, decline WITHOUT calling Gemma. Tune it using your test set.
MEMORY_TURNS = 3    # how many previous question/answer pairs are remembered for follow-ups
EXAMPLE_QUESTIONS = [   # EDIT THESE: use questions your documents really answer
    "What is the minimum attendance required to sit the end-semester examination?",
    "Can I apply for re-evaluation, and what is the deadline?",
    "What happens if my attendance is below the minimum?",
    "Who won the 2022 World Cup?",           # keep one question the documents cannot answer
]

RULES_V3 = """You answer students' questions using ONLY the context passages provided.
- Each passage begins with its source, for example [Exam Regulations.pdf, p. 4].
- After each fact you use, cite the source in brackets, for example (Exam Regulations.pdf, p. 4).
- If the answer is not in the context, reply exactly: "The answer could not be found in the documents."
- If passages from DIFFERENT documents give different rules or numbers for the same thing (for example two different attendance percentages), start your reply with the single word CONFLICT: then state both versions and cite both. Do not choose between them.
- Otherwise never write the word CONFLICT.
- Keep answers concise: 2 to 4 sentences."""

REWRITE_RULES = """You rewrite a student's latest question so that it can be understood without the earlier conversation.
- Use the conversation only to resolve words like "it", "that", "those" or "what about hostel?".
- If the question is already clear on its own, return it unchanged.
- Return ONLY the rewritten question, with no explanation."""

PLAIN_RULES = "Answer the student's question about college rules in 2 to 4 sentences."


# ---------- Reading files (PDF, Markdown, text) ----------
def read_file(path):
    """Returns a list of page texts. Markdown and text files count as one page."""
    if path.lower().endswith(".pdf"):
        return read_pdf(path)                      # defined in Step 4
    return [open(path, encoding="utf-8", errors="ignore").read()]


def add_documents(files):
    """Reads every uploaded file, reports a result for EACH file, and rebuilds the knowledge base."""
    if not files:
        return "No files uploaded yet."
    all_chunks, report = [], []
    for path in files:
        name = os.path.basename(path)
        try:
            pages = read_file(path)
            chunks = split_into_chunks(pages)      # defined in Step 4
            if not chunks:
                raise ValueError("no readable text (scanned image?)")
            for c in chunks:
                c["doc"] = name
            all_chunks += chunks
            report.append(f"✅ **{name}**: {len(pages)} pages, {len(chunks)} chunks")
        except Exception as e:
            report.append(f"❌ **{name}**: {type(e).__name__}: {e}")   # the real error stays visible here
    if all_chunks:
        vectors = embedder.encode([c["text"] for c in all_chunks], normalize_embeddings=True)
        knowledge_base.update(name=f"{len(files)} documents", chunks=all_chunks, vectors=vectors)
    return "\n\n".join(report)


# ---------- The pipeline: threshold, document names, conflict marker ----------
def answer_v2(question):
    """Same name as before, so any test code you wrote still works."""
    if not knowledge_base["chunks"]:
        return "Upload at least one document first.", []
    found = retrieve(question)                     # defined in Step 4
    if found[0]["score"] < MIN_SCORE:              # nothing relevant: do not call Gemma at all
        return f"The answer could not be found in the documents. Please contact {CONTACT_HINT}.", found
    context = "\n\n".join(f"[{c['doc']}, p. {c['page']}] {c['text']}" for c in found)
    reply = generate(f"Context:\n{context}\n\nQuestion: {question}", system_text=RULES_V3,
                     demo_reply=f"**Demonstration mode:** no API key. Best match: {found[0]['doc']}, "
                                f"page {found[0]['page']}:\n\n> {found[0]['text'][:400]}...")
    if reply.startswith("The answer could not be found"):
        reply += f" Please contact {CONTACT_HINT}."
    return reply, found


# ---------- Conversation memory ----------
def rewrite_question(question, history):
    """Turns a follow-up like 'and for labs?' into a standalone question before retrieval."""
    if not history:
        return question
    convo = "\n".join(f"Student: {q}\nAssistant: {a[:300]}" for q, a in history)
    out = generate(f"Conversation so far:\n{convo}\n\nLatest question: {question}",
                   system_text=REWRITE_RULES, demo_reply=question)
    out = (out or "").strip().strip('"')
    return out if out and len(out) < 300 else question


# ---------- RAG versus no RAG ----------
def plain_answer(question):
    """Asks Gemma the same question with NO documents, to show why retrieval matters."""
    return generate(question, system_text=PLAIN_RULES,
                    demo_reply="Demonstration mode: no API key, so there is no plain-model answer.")


# ---------- Interface ----------
def on_ask(question, history, compare):
    if not question or not question.strip():
        raise gr.Error("Type a question first.")
    history = history or []
    standalone = rewrite_question(question, history)
    reply, found = answer_v2(standalone)

    conflict = reply.startswith("CONFLICT:")
    if conflict:
        reply = reply[len("CONFLICT:"):].strip()
    refused = reply.startswith("The answer could not be found")

    html = ""
    if conflict:
        html += ("<div class='conflict'>⚠️ <b>The documents give conflicting information.</b> "
                 "Both versions are shown below. Please confirm with "
                 f"{CONTACT_HINT}.</div>")
    if standalone.strip() != question.strip():
        html += f"<div class='note'>Searched for: <i>{standalone}</i></div>"
    if found and found[0]["score"] >= MIN_SCORE and not refused:
        seen = []
        for c in found:
            label = f"{c['doc']} · p. {c['page']}"
            if label not in seen:
                seen.append(label)
        html += "<div class='chips'>" + "".join(f"<span class='chip'>{s}</span>" for s in seen) + "</div>"

    sources = "\n\n".join(
        f"**{c['doc']}, page {c['page']}** (similarity {c['score']:.2f})\n> {c['text']}" for c in found
    ) or "No passages retrieved."

    plain = ""
    if compare:
        plain = "### Same question WITHOUT the documents (plain Gemma)\n" + plain_answer(standalone)

    history = (history + [(question, reply)])[-MEMORY_TURNS:]
    return reply, html, sources, plain, history


def on_clear():
    return [], "", "", "", "", ""


CSS = """
.gradio-container {max-width: 880px !important; margin: auto;}
.chips {display: flex; flex-wrap: wrap; gap: 8px; margin: 4px 0 12px;}
.chip {background: #E6F2F0; color: #0B5D55; border: 1px solid #B7D9D4; border-radius: 6px;
       padding: 3px 10px; font-size: 0.85rem;}
.conflict {background: #FFF4E5; color: #7A4100; border: 1px solid #F0C27B; border-radius: 6px;
           padding: 8px 12px; margin: 8px 0;}
.note {color: #5B6B7A; font-size: 0.85rem; margin: 4px 0;}
#ask-btn {font-size: 1.05rem;}
"""
THEME = gr.themes.Soft(primary_hue="teal", neutral_hue="slate",
                       font=[gr.themes.GoogleFont("IBM Plex Sans"), "system-ui", "sans-serif"])

gr.close_all()
with gr.Blocks(title=f"{COLLEGE_NAME} Regulations Assistant") as demo:
    gr.Markdown(f"# {COLLEGE_NAME} Regulations Assistant\n"
                "Ask about attendance, exams, hostel and library rules. "
                "Every answer cites the document and page it came from. "
                "You can ask follow-up questions.")
    history_state = gr.State([])
    with gr.Accordion("1. Upload the official documents", open=True):
        files_in = gr.File(label="PDF, .md or .txt files", file_count="multiple",
                           file_types=[".pdf", ".md", ".txt"], type="filepath")
        status_md = gr.Markdown("No documents loaded yet.")
    question_box = gr.Textbox(label="2. Your question", lines=2,
                              placeholder="For example: What is the minimum attendance for the end-semester exam?")
    compare_box = gr.Checkbox(label="Also show the answer without the documents (RAG vs no RAG)", value=False)
    with gr.Row():
        ask_btn = gr.Button("Get answer", variant="primary", elem_id="ask-btn", scale=3)
        clear_btn = gr.Button("New conversation", scale=1)
    gr.Examples(EXAMPLE_QUESTIONS, inputs=question_box, label="Try an example")
    answer_md = gr.Markdown()
    chips_html = gr.HTML()
    plain_md = gr.Markdown()
    with gr.Accordion("Passages used for this answer", open=True):
        sources_md = gr.Markdown()

    outs = [answer_md, chips_html, sources_md, plain_md, history_state]
    files_in.change(add_documents, inputs=files_in, outputs=status_md)
    ask_btn.click(on_ask, inputs=[question_box, history_state, compare_box], outputs=outs)
    question_box.submit(on_ask, inputs=[question_box, history_state, compare_box], outputs=outs)
    clear_btn.click(on_clear, outputs=[history_state, question_box, answer_md, chips_html, sources_md, plain_md])

demo.launch(share=True, theme=THEME, css=CSS)

Closing server running on port: 7860
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6b142a811099ff1b11.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# New section

# ============================================================
# REPLACEMENT FOR STEP 5 (paste into a NEW cell at the bottom,
# or replace the old Step 5 cell). Run Steps 0-4 first.
# Campus Regulations Assistant: multiple documents, citations
# with document names, relevance threshold, cleaner interface.
# ============================================================
import os
import numpy as np
import gradio as gr

# ---------- STUDENT TODO: your settings ----------
COLLEGE_NAME = "Your College Name"          # shown in the title
CONTACT_HINT = "the Examination Office or your department office"   # who to contact when an answer is not found
MIN_SCORE = 0.20    # below this similarity, decline WITHOUT calling Gemma. Tune it using your 15-question test.
EXAMPLE_QUESTIONS = [
    "What is the minimum attendance required to sit the end-semester examination?",
    "Can I apply for re-evaluation, and what is the deadline?",
    "What are the hostel visiting hours?",
    "Who won the 2022 World Cup?",           # keep one question the documents cannot answer
]

RULES_V2 = """You answer students' questions using ONLY the context passages provided.
- Each passage begins with its source, for example [Exam Regulations.pdf, p. 4].
- After each fact you use, cite the source in brackets, for example (Exam Regulations.pdf, p. 4).
- If the answer is not in the context, reply exactly: "The answer could not be found in the documents."
- If two passages disagree, mention both and cite both.
- Keep answers concise: 2 to 4 sentences."""


# ---------- Reading files (PDF, Markdown, text) ----------
def read_file(path):
    """Returns a list of page texts. Markdown and text files count as one page."""
    if path.lower().endswith(".pdf"):
        return read_pdf(path)                      # defined in Step 4
    return [open(path, encoding="utf-8", errors="ignore").read()]


def add_documents(files):
    """Reads every uploaded file, reports a result for EACH file, and rebuilds the knowledge base."""
    if not files:
        return "No files uploaded yet."
    all_chunks, report = [], []
    for path in files:
        name = os.path.basename(path)
        try:
            pages = read_file(path)
            chunks = split_into_chunks(pages)      # defined in Step 4
            if not chunks:
                raise ValueError("no readable text (scanned image?)")
            for c in chunks:
                c["doc"] = name
            all_chunks += chunks
            report.append(f"✅ **{name}**: {len(pages)} pages, {len(chunks)} chunks")
        except Exception as e:
            report.append(f"❌ **{name}**: {type(e).__name__}: {e}")   # the real error stays visible here
    if all_chunks:
        vectors = embedder.encode([c["text"] for c in all_chunks], normalize_embeddings=True)
        knowledge_base.update(name=f"{len(files)} documents", chunks=all_chunks, vectors=vectors)
    return "\n\n".join(report)


# ---------- The pipeline, with document names and a relevance threshold ----------
def answer_v2(question):
    if not knowledge_base["chunks"]:
        return "Upload at least one document first.", []
    found = retrieve(question)                     # defined in Step 4
    if found[0]["score"] < MIN_SCORE:              # nothing relevant: do not call Gemma at all
        return f"The answer could not be found in the documents. Please contact {CONTACT_HINT}.", found
    context = "\n\n".join(f"[{c['doc']}, p. {c['page']}] {c['text']}" for c in found)
    reply = generate(f"Context:\n{context}\n\nQuestion: {question}", system_text=RULES_V2,
                     demo_reply=f"**Demonstration mode:** no API key. Best match: {found[0]['doc']}, "
                                f"page {found[0]['page']}:\n\n> {found[0]['text'][:400]}...")
    if reply.startswith("The answer could not be found"):
        reply += f" Please contact {CONTACT_HINT}."
    return reply, found


# ---------- Interface ----------
def on_ask(question):
    if not question or not question.strip():
        raise gr.Error("Type a question first.")
    reply, found = answer_v2(question)
    chips = ""
    if found and found[0]["score"] >= MIN_SCORE:
        seen = []
        for c in found:
            label = f"{c['doc']} · p. {c['page']}"
            if label not in seen:
                seen.append(label)
        chips = "<div class='chips'>" + "".join(f"<span class='chip'>{s}</span>" for s in seen) + "</div>"
    sources = "\n\n".join(
        f"**{c['doc']}, page {c['page']}** (similarity {c['score']:.2f})\n> {c['text']}" for c in found
    ) or "No passages retrieved."
    return reply, chips, sources


CSS = """
.gradio-container {max-width: 880px !important; margin: auto;}
.chips {display: flex; flex-wrap: wrap; gap: 8px; margin: 4px 0 12px;}
.chip {background: #E6F2F0; color: #0B5D55; border: 1px solid #B7D9D4; border-radius: 6px;
       padding: 3px 10px; font-size: 0.85rem;}
#ask-btn {font-size: 1.05rem;}
"""
THEME = gr.themes.Soft(primary_hue="teal", neutral_hue="slate",
                       font=[gr.themes.GoogleFont("IBM Plex Sans"), "system-ui", "sans-serif"])

gr.close_all()
with gr.Blocks(title=f"{COLLEGE_NAME} Regulations Assistant") as demo:
    gr.Markdown(f"# {COLLEGE_NAME} Regulations Assistant\n"
                "Ask about attendance, exams, hostel and library rules. "
                "Every answer cites the document and page it came from.")
    with gr.Accordion("1. Upload the official documents", open=True):
        files_in = gr.File(label="PDF, .md or .txt files", file_count="multiple",
                           file_types=[".pdf", ".md", ".txt"], type="filepath")
        status_md = gr.Markdown("No documents loaded yet.")
    question_box = gr.Textbox(label="2. Your question", lines=2,
                              placeholder="For example: What is the minimum attendance for the end-semester exam?")
    ask_btn = gr.Button("Get answer", variant="primary", elem_id="ask-btn")
    gr.Examples(EXAMPLE_QUESTIONS, inputs=question_box, label="Try an example")
    answer_md = gr.Markdown()
    chips_html = gr.HTML()
    with gr.Accordion("Passages used for this answer", open=False):
        sources_md = gr.Markdown()

    files_in.change(add_documents, inputs=files_in, outputs=status_md)
    ask_btn.click(on_ask, inputs=question_box, outputs=[answer_md, chips_html, sources_md])
    question_box.submit(on_ask, inputs=question_box, outputs=[answer_md, chips_html, sources_md])

demo.launch(share=True, theme=THEME, css=CSS)